# MinerU `text_level` Refinement Agent

**Purpose**: Correct `text_level` tags in MinerU's `_content_list.json` output using Gemini 2.5 Flash VLM-based TOC extraction.

**Pipeline**:
1. **VLM TOC Extraction** — Send PDF to Gemini 2.5 Flash → extract section headings as structured JSON
2. **Human-in-the-Loop Review** — Display & confirm the LLM's TOC detection result before proceeding
3. **Fuzzy Match & Correct** — Use confirmed ground truth to fix `text_level` in `content_list.json`

**Dependencies**: `google-genai`, `rapidfuzz`, `PyMuPDF (fitz)`

In [1]:
import json
import os
import re
from pathlib import Path

from IPython.display import display, Markdown
from google import genai
from google.genai import types
from rapidfuzz import fuzz

# ============================================================
# CONFIGURATION — Edit these variables before running
# ============================================================
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
GEMINI_MODEL = 'gemini-2.5-flash'

DOC_CODE = "SG_Personal Data Protection Act 2012"
PDF_PATH  = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\data privacy\{DOC_CODE}.pdf"
JSON_PATH = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\{DOC_CODE}\hybrid_auto\{DOC_CODE}_content_list.json"

OUTPUT_DIR = None  # Set to a directory path, or None to save alongside input JSON

# Human-in-the-loop temp file
HITL_JSON_PATH = None  # Auto-generated from JSON_PATH if None

print('Configuration loaded')
print(f'  Model: {GEMINI_MODEL}')
print(f'  PDF:   {PDF_PATH}')
print(f'  JSON:  {JSON_PATH}')

Configuration loaded
  Model: gemini-2.5-flash
  PDF:   C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\data privacy\SG_Personal Data Protection Act 2012.pdf
  JSON:  C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\SG_Personal Data Protection Act 2012\hybrid_auto\SG_Personal Data Protection Act 2012_content_list.json


In [3]:
# ============================================================
# Load content_list.json and show statistics
# ============================================================
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    content_list = json.load(f)

total_blocks = len(content_list)
text_level_blocks = [b for b in content_list if 'text_level' in b]
pages = set(b.get('page_idx', -1) for b in content_list)

print(f'Content List Statistics')
print(f'  Total blocks:          {total_blocks}')
print(f'  Blocks with text_level: {len(text_level_blocks)}')
print(f'  Page range:            {min(pages)} - {max(pages)}')
print(f'\n--- Current text_level blocks ---')
for i, b in enumerate(text_level_blocks):
    print(f'  [{i+1:3d}] page {b["page_idx"]:3d} | {b.get("text", "")[:80]}')

Content List Statistics
  Total blocks:          2041
  Blocks with text_level: 250
  Page range:            0 - 123

--- Current text_level blocks ---
  [  1] page   0 | THE STATUTES OF THE REPUBLIC OF SINGAPORE 
  [  2] page   0 | PERSONAL DATA PROTECTION ACT 2012 
  [  3] page   1 | Personal Data Protection Act 2012 
  [  4] page   1 | Section 
  [  5] page   2 | PART 4 
  [  6] page   2 | COLLECTION, USE AND DISCLOSURE OF PERSONAL DATA 
  [  7] page   2 | Section 
  [  8] page   2 | Division 2 — Purpose 
  [  9] page   2 | PART 5 
  [ 10] page   2 | ACCESS TO AND CORRECTION OF PERSONAL DATA 
  [ 11] page   2 | PART 6 
  [ 12] page   2 | CARE OF PERSONAL DATA 
  [ 13] page   2 | PART 6A 
  [ 14] page   2 | NOTIFICATION OF DATA BREACHES 
  [ 15] page   3 | PART 7 
  [ 16] page   3 | Section 
  [ 17] page   3 | PART 8 
  [ 18] page   3 | PART 9 
  [ 19] page   3 | DO NOT CALL REGISTRY 
  [ 20] page   3 | Division 1 — Preliminary 
  [ 21] page   3 | Division 2 — Administration 
  [ 22]

## Stage 1: VLM TOC Extraction

Send the PDF document to **Gemini 2.5 Flash** to extract:
1. Which pages contain the Table of Contents (`toc_pages`, 0-indexed)
2. All **top-level** section headings listed in the TOC (`sections`)

The model receives the native PDF bytes and returns structured JSON.

In [4]:
# ============================================================
# Stage 1: VLM TOC Extraction using Gemini 2.5 Flash
# ============================================================

# Initialize Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

# Read PDF bytes
with open(PDF_PATH, 'rb') as f:
    pdf_bytes = f.read()
print(f'PDF loaded: {len(pdf_bytes):,} bytes')

# System prompt for TOC extraction
SYSTEM_PROMPT = """
You are a Document Structure Analyst. Your task is to identify and extract the structural hierarchy from a document's Table of Contents (TOC).

RULES:
- Assign "level": 1 to the TOP-MOST heading tier in this document (e.g. CHAPTER, PART, TITLE, SECTION — whatever is the highest level).
- Assign "level": 2 to the SECOND tier headings that fall under level 1 (e.g. Article, Section, Clause).
- DO NOT include level 3 or below (sub-sections, sub-clauses, bullet points, indented descriptions).
- EXCLUDE page numbers, dot leaders (....), and extra whitespace.
- If the heading format in the TOC differs from the body content (e.g. TOC uses "Section 1 title" but body says "1. title"), ALWAYS  the exact wording and format as it appears in the body content, not the TOC.
- "heading": the structural label ONLY (e.g. "CHAPTER I", "Article 1", "Section 2", "1").
- "heading_name": the descriptive title ONLY (e.g. "General provisions"). If none exists, use null.
- "page": the page number that the section actually appears on, Not the TOC page number. If no page number is listed in TOC, use null.

Output format:
{
  "sections": [
    {"level": 1, "heading": "CHAPTER I",  "heading_name": "General provisions",          "page": 3},
    {"level": 2, "heading": "Article 1",  "heading_name": "Subject-matter and objectives","page": 3},
    {"level": 1, "heading": "CHAPTER II", "heading_name": "Principles",                   "page": 7},
    {"level": 2, "heading": "Article 5",  "heading_name": null,                           "page": 7}
  ]
}

Only output valid JSON.
"""

# Call Gemini 2.5 Flash
try:
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
            'Extract the Table of Contents structure from this PDF document. Return the result as JSON.'
        ],
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type='application/json',
            temperature=0,
            thinking_config=types.ThinkingConfig(thinking_budget=0)
        )
    )
except Exception as e:
    print(f'Error during Gemini API call: {e}')
    raise

# Parse result
raw_text = response.text
toc_result = json.loads(raw_text)

print(f'\nVLM TOC Extraction Complete')
print(f'  Sections found: {len(toc_result.get("sections", []))}')
print(f'\n--- Extracted Sections ---')
print(json.dumps(toc_result, indent=2))

PDF loaded: 461,395 bytes

VLM TOC Extraction Complete
  Sections found: 116

--- Extracted Sections ---
{
  "sections": [
    {
      "level": 1,
      "heading": "PART 1",
      "heading_name": "PRELIMINARY",
      "page": 7
    },
    {
      "level": 2,
      "heading": "1.",
      "heading_name": "Short title",
      "page": 7
    },
    {
      "level": 2,
      "heading": "2.",
      "heading_name": "Interpretation",
      "page": 7
    },
    {
      "level": 2,
      "heading": "3.",
      "heading_name": "Purpose",
      "page": 13
    },
    {
      "level": 2,
      "heading": "4.",
      "heading_name": "Application of Act",
      "page": 13
    },
    {
      "level": 1,
      "heading": "PART 2",
      "heading_name": "PERSONAL DATA PROTECTION COMMISSION AND ADMINISTRATION",
      "page": 14
    },
    {
      "level": 2,
      "heading": "5.",
      "heading_name": "Personal Data Protection Commission",
      "page": 14
    },
    {
      "level": 2,
      "heading": "6

In [14]:
# ============================================================
# Human-in-the-Loop: Review & Confirm TOC Result
# ============================================================

# Determine save path for the ground truth JSON
if HITL_JSON_PATH is None:
    json_dir = Path(JSON_PATH).parent
    HITL_JSON_PATH = str(json_dir / 'toc_ground_truth.json')

# Save LLM result to file for review / editing
with open(HITL_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(toc_result, f, indent=2, ensure_ascii=False)

print(f'Saved TOC result to: {HITL_JSON_PATH}')
print(f'\n{"="*70}')
print(f'LLM TOC Detection Result (review below)')
print(f'{"="*70}')
print(f'\nSections ({len(toc_result["sections"])} total):')
print(f'{"-"*70}')
for i, s in enumerate(toc_result['sections'], 1):
    sid = s.get('heading') or 'PART'
    print(f' {sid} {s["heading_name"]}')
print(f'{"-"*70}')

# ---- Confirmation gate ----
print(f'\nREVIEW the result above.')
print(f'If you need to edit, open the file at:\n  {HITL_JSON_PATH}')
confirm = input('\nType "yes" to confirm, or "edit" if you edited the file: ').strip().lower()

if confirm in ('edit', 'e'):
    # Reload the user-edited file
    with open(HITL_JSON_PATH, 'r', encoding='utf-8') as f:
        toc_result = json.load(f)
    print(f'\nReloaded edited file. Sections: {len(toc_result["sections"])}')
    # Show updated result
    for i, s in enumerate(toc_result['sections'], 1):
        sid = s.get('section_id') or ''
        print(f' {i:3d}. [{sid:>5}] {s["heading_name"]}')
elif confirm in ('yes', 'y'):
    print('\nConfirmed! Proceeding with correction...')
else:
    raise RuntimeError(f'Aborted. Got "{confirm}". Re-run this cell after review.')

# Store confirmed ground truth for Stage 2
# toc_pages = set(toc_result.get('toc_pages', []))
ground_truth_sections = toc_result['sections']
print(f'\nGround Truth: {len(ground_truth_sections)} sections')    

Saved TOC result to: C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\SG_Personal Data Protection Act 2012\hybrid_auto\toc_ground_truth.json

LLM TOC Detection Result (review below)

Sections (116 total):
----------------------------------------------------------------------
 PART 1 PRELIMINARY
 1. Short title
 2. Interpretation
 3. Purpose
 4. Application of Act
 PART 2 PERSONAL DATA PROTECTION COMMISSION AND ADMINISTRATION
 5. Personal Data Protection Commission
 6. Functions of Commission
 7. Advisory committees
 8. Delegation
 9. Conduct of proceedings
 10. Cooperation agreements
 PART 3 GENERAL RULES WITH RESPECT TO PROTECTION OF AND ACCOUNTABILITY FOR PERSONAL DATA
 11. Compliance with Act
 12. Policies and practices
 PART 4 COLLECTION, USE AND DISCLOSURE OF PERSONAL DATA
 13. Consent required
 14. Provision of consent
 15. Deemed consent
 15A. Deemed consent by notification
 16. Withdrawal of consent
 17. Collection, use and disclo

## Stage 2: Fuzzy Match & Correct `content_list.json`

Using the confirmed ground truth, apply corrections:

1. **Remove false positives** — Body blocks with `text_level: 1` that don't match any ground truth section
2. **Add false negatives** — Body blocks that match ground truth but currently lack `text_level`

In [15]:
# ============================================================
# Stage 2: Fuzzy Matching & Correction Engine
# ============================================================

# Higher threshold since ground truth titles MUST exist in the body text
# MinerU errors are layout misclassification, not content errors
FUZZY_THRESHOLD = 95     # strict: titles should match very closely
# TOKEN_SET_THRESHOLD = 92  # for token-set ratio pass (handles minor OCR gaps)
# MIN_LEN_RATIO = 0.7       # block text length must be >= 70% of target length

def normalize(text):
    """Normalize text for comparison: lowercase, strip, collapse whitespace,
    remove trailing dots and page numbers."""
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text


def build_match_targets(sections):
    """Build match candidates with exact page constraints.
    
    toc_page → 對應 content_list 的 page_idx (0-indexed)
    要求 exact match：block_page 必須等於 toc_page - 1
    """
    heading_targets = []
    full_targets = []

    level1 = [s for s in sections if s.get('level') == 1]
    use_sections = level1 if level1 else [s for s in sections if s.get('level') == 2]

    for idx, s in enumerate(use_sections):
        heading      = s.get('heading', '') or ''
        heading_name = s.get('heading_name', '') or ''
        orig_title   = f"{heading} {heading_name}".strip()
        toc_page     = s.get('page')   # LLM 返回，1-indexed

        # Exact page match: page_range = (exact_page, exact_page)
        # 如果 toc_page 是 None，page_range = None（不作 page 限制）
        if toc_page is not None:
            exact_page = toc_page - 1   # 0-indexed
            page_range = (exact_page, exact_page)
        else:
            page_range = None   # no page constraint

        if heading:
            heading_targets.append((normalize(heading), orig_title, page_range))
        if heading and heading_name:
            full_targets.append((normalize(f'{heading} {heading_name}'), orig_title, page_range))

    return heading_targets, full_targets


def match_block(block_text, block_page, heading_targets, full_targets):
    """Check if a block's text matches any ground truth section.
    Pass 1: heading only     — e.g. block = "Section 2"
    Pass 2: heading + name   — e.g. block = "Section 2 Customer acceptance policy"
    Returns (matched, orig_title, score, method)
    
    條件 1：fuzzy score >= FUZZY_THRESHOLD
    條件 2：block_page == exact_page (exact match)
    兩個條件都必須滿足。
    """
    norm_text = normalize(block_text)
    if not norm_text:
        return False, None, 0, None

    def page_ok(page_range):
        """Check if block_page exactly matches the page."""
        if page_range is None:
            return True   # no constraint
        return block_page == page_range[0]  # exact match

    best_score = 0
    best_title = None

    # Pass 1: heading only
    for norm_target, orig_title, page_range in heading_targets:
        score = fuzz.ratio(norm_text, norm_target)
        if score > best_score:
            best_score = score
            best_title = orig_title
        if score >= FUZZY_THRESHOLD and page_ok(page_range):
            return True, orig_title, score, 'heading'

    # Pass 2: heading + heading_name (only if Pass 1 failed)
    for norm_target, orig_title, page_range in full_targets:
        score = fuzz.ratio(norm_text, norm_target)
        if score > best_score:
            best_score = score
            best_title = orig_title
        if score >= FUZZY_THRESHOLD and page_ok(page_range):
            return True, orig_title, score, 'heading+name'

    return False, best_title, best_score, None


# Build match targets from confirmed ground truth
heading_targets, full_targets = build_match_targets(ground_truth_sections)

# Deep copy for correction
corrected = json.loads(json.dumps(content_list))
corrections = {'added': [], 'removed': [], 'kept': [], 'toc_removed': []}

for i, block in enumerate(corrected):
    page = block.get('page_idx', -1)
    page = int(page)              # 保持 0-indexed 給 page_ok() 用
    page_display = page + 1       # 只用於顯示

    has_level = 'text_level' in block
    block_text = block.get('text', '')
    block_type = block.get('type', '')

    # Skip non-text block types (image, table, page_number, header, etc.)
    if block_type not in ('text',):
        if has_level:
            del block['text_level']
            corrections['removed'].append({
                'index': i, 'page': page_display, 'text': block_text[:80],
                'reason': f'non-text block type: {block_type}'
            })
        continue
    
    # Body page blocks: match against ground truth
    matched, match_title, score, method = match_block(
        block_text, page, heading_targets, full_targets
    )

    if has_level and matched:
        # Correct — keep text_level
        corrections['kept'].append({
            'index': i, 'page': page_display, 'text': block_text[:80],
            'matched': match_title, 'score': score, 'method': method
        })
    elif has_level and not matched:
        # False positive — MinerU incorrectly tagged this as heading
        del block['text_level']
        corrections['removed'].append({
            'index': i, 'page': page_display, 'text': block_text[:80],
            'reason': f'no ground truth match (best score: {score})'
        })
    elif not has_level and matched:
        # False negative — MinerU missed this heading
        block['text_level'] = 1
        corrections['added'].append({
            'index': i, 'page': page_display, 'text': block_text[:80],
            'matched': match_title, 'score': score, 'method': method
        })

# ---- Summary ----
print('Correction Complete')
print('=' * 70)
print(f'  Kept (correct):        {len(corrections["kept"])}')
print(f'  Removed (false pos):   {len(corrections["removed"])}')
print(f'  Removed (TOC pages):   {len(corrections["toc_removed"])}')
print(f'  Added (false neg):     {len(corrections["added"])}')
print('=' * 70)

if corrections['removed']:
    print('\n--- Removed (False Positives) ---')
    for c in corrections['removed']:
        print(f'  page {c["page"]:3d} | {c["text"][:60]} | reason: {c["reason"]}')

if corrections['toc_removed']:
    print('\n--- Removed (TOC Page Entries) ---')
    for c in corrections['toc_removed']:
        print(f'  page {c["page"]:3d} | {c["text"][:60]}')

if corrections['added']:
    print('\n--- Added (False Negatives) ---')
    for c in corrections['added']:
        print(f'  page {c["page"]:3d} | {c["text"][:60]} | matched: {c["matched"]} ({c["score"]} via {c["method"]})')

Correction Complete
  Kept (correct):        26
  Removed (false pos):   224
  Removed (TOC pages):   0
  Added (false neg):     0

--- Removed (False Positives) ---
  page   1 | THE STATUTES OF THE REPUBLIC OF SINGAPORE  | reason: no ground truth match (best score: 44.44444444444444)
  page   1 | PERSONAL DATA PROTECTION ACT 2012  | reason: no ground truth match (best score: 57.446808510638306)
  page   2 | Personal Data Protection Act 2012  | reason: no ground truth match (best score: 57.446808510638306)
  page   2 | Section  | reason: no ground truth match (best score: 45.45454545454546)
  page   3 | PART 4  | reason: no ground truth match (best score: 100.0)
  page   3 | COLLECTION, USE AND DISCLOSURE OF PERSONAL DATA  | reason: no ground truth match (best score: 93.06930693069306)
  page   3 | Section  | reason: no ground truth match (best score: 45.45454545454546)
  page   3 | Division 2 — Purpose  | reason: no ground truth match (best score: 42.85714285714286)
  page   3 | PART 

In [16]:
print(fuzz.ratio(normalize("SEVENTH SCHEDULE CONSTITUTION AND PROCEEDINGS OF DATA PROTECTION APPEALPANEL AND DATA PROTECTION APPEAL COMMITTEES"), normalize("CONSTITUTION AND PROCEEDINGS OF DATA PROTECTION APPEALPANEL AND DATA PROTECTION APPEAL COMMITTEES")))

91.9431279620853


In [17]:
# ============================================================
# Save corrected JSON and correction log
# ============================================================

# Overwrite original file directly (avoid long path issue)
original_path = Path(JSON_PATH)
output_path = original_path.parent / (original_path.stem + '_corrected' + original_path.suffix)
log_path = Path(JSON_PATH).parent / 'correction_log.json'

# Save corrected content_list (overwrite original)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(corrected, f, indent=4, ensure_ascii=False)

# Save correction log for auditability
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump({
        'source': str(JSON_PATH),
        'ground_truth': str(HITL_JSON_PATH),
        'stats': {
            'total_blocks': len(corrected),
            'kept': len(corrections['kept']),
            'removed_false_pos': len(corrections['removed']),
            'removed_toc': len(corrections['toc_removed']),
            'added_false_neg': len(corrections['added']),
        },
        'corrections': corrections,
    }, f, indent=2, ensure_ascii=False)

# Before vs After comparison
original_levels = sum(1 for b in content_list if 'text_level' in b)
corrected_levels = sum(1 for b in corrected if 'text_level' in b)

print(f'Saved corrected JSON:  {output_path}')
print(f'Saved correction log:  {log_path}')
print(f'\nBefore -> After:')
print(f'  text_level blocks: {original_levels} -> {corrected_levels}')
print(f'  Net change: {corrected_levels - original_levels:+d}')

Saved corrected JSON:  C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\SG_Personal Data Protection Act 2012\hybrid_auto\SG_Personal Data Protection Act 2012_content_list_corrected.json
Saved correction log:  C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\SG_Personal Data Protection Act 2012\hybrid_auto\correction_log.json

Before -> After:
  text_level blocks: 250 -> 26
  Net change: -224


In [18]:
# ============================================================
# Validation Spot-Check
# ============================================================

# Show all text_level blocks in the corrected output
final_levels = [b for b in corrected if 'text_level' in b]

print(f'Final text_level Blocks ({len(final_levels)} total)')
print('=' * 70)
for i, b in enumerate(final_levels, 1):
    page = b.get('page_idx', '?')
    text = b.get('text', '')[:70]
    print(f'  {i:3d}. page {page:3d} | {text}')
print('=' * 70)

# Cross-check: which ground truth sections were NOT matched in the document?
matched_titles = set()
for c in corrections['kept'] + corrections['added']:
    matched_titles.add(c.get('matched', ''))

unmatched = [s for s in ground_truth_sections if s['heading_name'] not in matched_titles]

if unmatched:
    print(f'\nWARNING: Ground truth sections NOT found in content_list.json ({len(unmatched)}):')
    for s in unmatched:
        print(f'{s["heading_name"]}')
else:
    print(f'\nAll ground truth sections matched in content_list.json')

Final text_level Blocks (26 total)
    1. page   6 | PART 1 
    2. page  13 | PART 2 
    3. page  18 | PART 3 
    4. page  19 | PART 4 
    5. page  28 | PART 5 
    6. page  31 | PART 6 
    7. page  33 | PART 6A 
    8. page  37 | PART 7 
    9. page  37 | PART 8 
   10. page  37 | PART 9 
   11. page  50 | PART 9A 
   12. page  53 | PART 9B 
   13. page  60 | PART 9C 
   14. page  73 | PART 9D 
   15. page  76 | PART 10 
   16. page  94 | FIRST SCHEDULE 
   17. page 104 | SECOND SCHEDULE 
   18. page 107 | THIRD SCHEDULE 
   19. page 107 | FOURTH SCHEDULE 
   20. page 107 | FIFTH SCHEDULE 
   21. page 108 | SIXTH SCHEDULE 
   22. page 109 | SEVENTH SCHEDULE 
   23. page 112 | EIGHTH SCHEDULE
   24. page 113 | NINTH SCHEDULE 
   25. page 119 | TENTH SCHEDULE 
   26. page 119 | ELEVENTH SCHEDULE 

PRELIMINARY
Short title
Interpretation
Purpose
Application of Act
PERSONAL DATA PROTECTION COMMISSION AND ADMINISTRATION
Personal Data Protection Commission
Functions of Commission
Adviso